# VibeVoice — Peter & Stewie Voice Cloning

**Como usar:**
1. Certifica-te que o runtime é **GPU T4** (Runtime → Change runtime type → T4 GPU)
2. Corre as células em ordem (Shift+Enter ou Run All)
3. Na célula 3, faz upload dos ficheiros `peter_clean.wav` e `stewie_clean.wav`
4. No final, descarrega o ficheiro `peter_stewie_output.wav`


In [ ]:
# Célula 1: Instalar dependências (~3-5 min na primeira vez)
import subprocess
import sys

print('A clonar o repo VibeVoice...')
subprocess.run(['git', 'clone', 'https://github.com/microsoft/VibeVoice.git'], check=True)

print('A instalar dependências...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', './VibeVoice'], check=True)

# Tentar instalar flash-attn para melhor qualidade (pode falhar — tudo bem)
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'flash-attn', '--no-build-isolation'], 
                   check=True, timeout=300)
    print('flash-attn instalado — qualidade máxima')
except Exception as e:
    print(f'flash-attn não instalado ({e}) — a usar SDPA (funciona bem na mesma)')

print('\nDependências instaladas!')

In [ ]:
# Célula 2: Upload dos ficheiros de voz
import os
import shutil
from google.colab import files

voices_dir = './VibeVoice/demo/voices'
os.makedirs(voices_dir, exist_ok=True)

print('FAZ UPLOAD dos ficheiros de voz:')
print('  → peter_clean.wav  (voz do Peter Griffin)')
print('  → stewie_clean.wav (voz do Stewie Griffin)')
print('')
print('Podes fazer upload dos dois ao mesmo tempo.')

uploaded = files.upload()

for filename, content in uploaded.items():
    dest = os.path.join(voices_dir, filename)
    with open(dest, 'wb') as f:
        f.write(content)
    print(f'  ✓ {filename} → {dest}')

print('\nVozes carregadas!')
print('Ficheiros na pasta voices:')
for f in os.listdir(voices_dir):
    if f.endswith('.wav'):
        print(f'  {f}')

In [ ]:
# Célula 3: Script de diálogo Peter/Stewie
# Podes editar aqui o diálogo que queres gerar

script_text = """Speaker 1: Remote work? That's basically stealing from your boss, Stewie.
Speaker 2: Fascinating. You've somehow managed to make telecommuting a moral issue.
Speaker 1: If you're home, you're watching TV. Everyone knows that.
Speaker 2: I produce more in two hours remotely than you manage in an entire week.
Speaker 1: That's because you cheat. You use your baby brain.
Speaker 2: My baby brain outperforms your adult brain by every measurable metric.
Speaker 1: See? This is why they should make everyone go back to the office.
Speaker 2: So you can micromanage incompetence in person? Riveting strategy, Peter."""

script_path = './peter_stewie_script.txt'
with open(script_path, 'w') as f:
    f.write(script_text)

print('Script guardado:')
print(script_text)
print(f'\n→ {script_path}')

In [ ]:
# Célula 4: Gerar audio (5-10 min em GPU T4)
import subprocess
import sys
import os

output_dir = './vibevoice_output'
os.makedirs(output_dir, exist_ok=True)

cmd = [
    sys.executable,
    './VibeVoice/demo/inference_from_file.py',
    '--model_path', 'microsoft/VibeVoice-1.5b',
    '--txt_path', './peter_stewie_script.txt',
    '--speaker_names', 'peter_clean', 'stewie_clean',
    '--output_dir', output_dir,
    '--device', 'cuda',
    '--cfg_scale', '1.3',
]

print('A gerar audio... (GPU T4, ~5-10 min)')
print('Comando:', ' '.join(cmd))
print('')

result = subprocess.run(cmd, capture_output=False, text=True)

if result.returncode == 0:
    print('\n✓ Geração concluída!')
else:
    print(f'\n✗ Erro (código {result.returncode})')

In [ ]:
# Célula 5: Descarregar output
import os
from google.colab import files

output_dir = './vibevoice_output'
wav_files = [f for f in os.listdir(output_dir) if f.endswith('.wav')]

print('Ficheiros gerados:')
for f in wav_files:
    path = os.path.join(output_dir, f)
    size_mb = os.path.getsize(path) / 1024 / 1024
    print(f'  {f}  ({size_mb:.1f} MB)')

print('\nA fazer download...')
for f in wav_files:
    files.download(os.path.join(output_dir, f))
    print(f'  ✓ {f}')

print('\nDone! Ouve o ficheiro e diz o que achas.')